# 01 Generate Synthetic Dealership Lead Dataset

This notebook creates a realistic synthetic CRM-style dataset for a car dealership lead conversion project.

The dataset includes customer enquiry details, vehicle interest, sales activity, follow-up behaviour and final conversion outcome.

In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

## Dataset Design

The dataset is designed to reflect common dealership CRM fields, including:

- Lead source
- Customer age group
- Vehicle type
- New, used or demo interest
- Finance enquiry
- Trade-in interest
- Test drive completion
- Quote provided
- Sales response time
- Follow-up count
- Final conversion outcome

In [2]:
n_leads = 1500

lead_ids = [f"L{str(i).zfill(4)}" for i in range(1, n_leads + 1)]

data = pd.DataFrame({
    "lead_id": lead_ids,
    "lead_source": np.random.choice(
        ["Website", "Walk-in", "Phone", "Carsales", "Referral", "Social Media"],
        size=n_leads,
        p=[0.25, 0.18, 0.16, 0.20, 0.11, 0.10]
    ),
    "customer_age_group": np.random.choice(
        ["18-24", "25-34", "35-44", "45-54", "55+"],
        size=n_leads,
        p=[0.12, 0.30, 0.27, 0.20, 0.11]
    ),
    "customer_location": np.random.choice(
        ["Local", "Nearby Suburb", "Far Suburb"],
        size=n_leads,
        p=[0.45, 0.35, 0.20]
    ),
    "vehicle_type": np.random.choice(
        ["SUV", "Sedan", "Hatchback", "Ute", "EV"],
        size=n_leads,
        p=[0.35, 0.20, 0.18, 0.17, 0.10]
    ),
    "new_or_used": np.random.choice(
        ["New", "Used", "Demo"],
        size=n_leads,
        p=[0.45, 0.40, 0.15]
    ),
    "budget_range": np.random.choice(
        ["Low", "Medium", "High"],
        size=n_leads,
        p=[0.35, 0.45, 0.20]
    ),
    "finance_enquiry": np.random.choice([0, 1], size=n_leads, p=[0.55, 0.45]),
    "trade_in": np.random.choice([0, 1], size=n_leads, p=[0.62, 0.38]),
    "test_drive_completed": np.random.choice([0, 1], size=n_leads, p=[0.58, 0.42]),
    "quote_provided": np.random.choice([0, 1], size=n_leads, p=[0.50, 0.50]),
    "previous_customer": np.random.choice([0, 1], size=n_leads, p=[0.72, 0.28]),
    "response_time_hours": np.random.gamma(shape=2.2, scale=5.0, size=n_leads).round(1),
    "follow_up_count": np.random.poisson(lam=3, size=n_leads),
    "days_since_enquiry": np.random.randint(1, 45, size=n_leads)
})

data.head()

,lead_id,lead_source,customer_age_group,customer_location,vehicle_type,new_or_used,budget_range,finance_enquiry,trade_in,test_drive_completed,quote_provided,previous_customer,response_time_hours,follow_up_count,days_since_enquiry
0,L0001,Walk-in,35-44,Nearby Suburb,Hatchback,Used,Medium,1,0,1,0,0,18.8,5,38
1,L0002,Social Media,35-44,Nearby Suburb,Sedan,New,High,0,0,0,1,1,4.0,2,20
2,L0003,Carsales,18-24,Local,SUV,Demo,Low,1,1,0,0,0,5.5,2,29
3,L0004,Carsales,25-34,Nearby Suburb,SUV,Used,Low,0,0,1,0,0,2.9,2,6
4,L0005,Website,25-34,Nearby Suburb,Sedan,New,Medium,1,1,0,0,0,7.2,2,9


## Conversion Logic

The conversion outcome is generated using a weighted probability model.

Leads are more likely to convert when they show stronger buying intent, such as completing a test drive, requesting finance, asking for a quote, having a trade-in or being a previous customer.

Leads are less likely to convert when response time is slow or the enquiry becomes old.

In [3]:
conversion_score = (
    -2.2
    + 1.4 * data["test_drive_completed"]
    + 0.8 * data["finance_enquiry"]
    + 0.6 * data["trade_in"]
    + 0.9 * data["quote_provided"]
    + 0.5 * data["previous_customer"]
    + 0.12 * data["follow_up_count"]
    - 0.035 * data["response_time_hours"]
    - 0.025 * data["days_since_enquiry"]
)

conversion_score += data["lead_source"].map({
    "Referral": 0.7,
    "Walk-in": 0.5,
    "Phone": 0.2,
    "Website": 0.0,
    "Carsales": -0.1,
    "Social Media": -0.3
})

conversion_score += data["budget_range"].map({
    "High": 0.25,
    "Medium": 0.15,
    "Low": -0.15
})

conversion_probability = 1 / (1 + np.exp(-conversion_score))

data["converted"] = np.random.binomial(1, conversion_probability)

data.head()

,lead_id,lead_source,customer_age_group,customer_location,vehicle_type,new_or_used,budget_range,finance_enquiry,trade_in,test_drive_completed,quote_provided,previous_customer,response_time_hours,follow_up_count,days_since_enquiry,converted
0,L0001,Walk-in,35-44,Nearby Suburb,Hatchback,Used,Medium,1,0,1,0,0,18.8,5,38,1
1,L0002,Social Media,35-44,Nearby Suburb,Sedan,New,High,0,0,0,1,1,4.0,2,20,1
2,L0003,Carsales,18-24,Local,SUV,Demo,Low,1,1,0,0,0,5.5,2,29,0
3,L0004,Carsales,25-34,Nearby Suburb,SUV,Used,Low,0,0,1,0,0,2.9,2,6,0
4,L0005,Website,25-34,Nearby Suburb,Sedan,New,Medium,1,1,0,0,0,7.2,2,9,0


In [4]:
print(f"Rows: {data.shape[0]}")
print(f"Columns: {data.shape[1]}")
print(f"Conversion rate: {data['converted'].mean():.2%}")

Rows: 1500
Columns: 16
Conversion rate: 34.27%


In [5]:
data["converted"].value_counts(normalize=True)

converted
0    0.657333
1    0.342667
Name: proportion, dtype: float64

In [6]:
output_path = "../data/raw/dealership_leads.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

data.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

Dataset saved to: ../data/raw/dealership_leads.csv
